# Order test

### Overview

Checks whether order of events matter

**Input:** the per-review event sequences from 3.1 (`checkpoints/event_sequences.jsonl`),
relabelled into the three granularities, and the nested-label map.

**Pipeline:**
1. Setup and load every granularity.
2. Test 1: bigram mutual information against a within-review shuffle null (each
   chain reshuffled in place, so its multiset and length are preserved and only order
   is destroyed).
3. Test 2: mean relative position per event type, and the spread across types.
4. Gap decay: repeat Test 1 at skip distances k = 0,1,2,3,5.
5. Transition lift: observed vs expected counts for each pair.
6. Save one summary row per granularity.

## 1. Setup

In [10]:
# Cell 1: Imports

import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats

In [11]:
# Cell 2: Parameters

CHECKPOINTS = Path("checkpoints")
OUT_DIR     = Path("outputs_event_chains/order")
OUT_DIR.mkdir(exist_ok=True)
SEQUENCES = CHECKPOINTS / "event_sequences.jsonl"
NESTED_PATH = Path("data/label_to_nested_mapping_reference.csv")
nested_df   = pd.read_csv(NESTED_PATH)
LABEL_TO_MR     = dict(zip(nested_df["Label"], nested_df["MR_nested"]))
LABEL_TO_AGENCY = dict(zip(nested_df["Label"], nested_df["Agency_nested"])) 


GRANULARITIES = [
    ("clusters_30",      SEQUENCES, None),
    ("MR_4",     SEQUENCES, LABEL_TO_MR),
    ("Agency_6", SEQUENCES, LABEL_TO_AGENCY),
]
RANDOM_STATE = 42
N_PERMUTATIONS = 200

# Test 2:
MIN_CHAIN_LEN   = 5
N_POSITION_BINS = 10

# Effect-size interpretation
CRAMERS_V_SMALL = 0.10

SUMMARY_PATH = OUT_DIR / "order_signal_summary.csv"

In [12]:
# Cell 3: Load event chains per alphabet

def load_chains(path, remap):
    with open(path) as f:
        seqs = [json.loads(line) for line in f]

    chains = []
    for s in seqs:
        tokens = [e["event"] for e in s["events"]]
        if remap is not None:
            missing = {t for t in tokens if t not in remap}
            assert not missing, f"tokens with no nested label: {missing}"
            tokens = [remap[t] for t in tokens]
        chains.append(tokens)
    return seqs, chains


loaded = {}
for name, path, remap in GRANULARITIES:
    seqs, chains = load_chains(path, remap)
    loaded[name] = {"seqs": seqs, "chains": chains}

    lengths = np.array([len(c) for c in chains])
    n_types = len({t for c in chains for t in c})
    print(f"{name:14s} {len(chains):,} reviews   {int(lengths.sum()):,} tokens   "
          f"{n_types:2d} types   median chain {int(np.median(lengths))}")

clusters_30    6,501 reviews   119,377 tokens   30 types   median chain 17
MR_4           6,501 reviews   119,377 tokens    4 types   median chain 17
Agency_6       6,501 reviews   119,377 tokens    6 types   median chain 17


## 2. Test 1: does the next event depend on the current one?

In [13]:
# Cell 4: Functions for computing order signal

def bigram_table(int_chains, n_types):
    tbl = np.zeros((n_types, n_types), dtype=np.int64)
    for c in int_chains:
        if len(c) < 2:
            continue
        np.add.at(tbl, (c[:-1], c[1:]), 1)
    return tbl

def mutual_information(tbl):
    total = tbl.sum()
    if total == 0:
        return 0.0
    p  = tbl / total
    px = p.sum(axis=1, keepdims=True)
    py = p.sum(axis=0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = p * np.log(p / (px * py))
    return float(np.nansum(terms))

def successor_entropy(tbl):
    py = tbl.sum(axis=0) / tbl.sum()
    py = py[py > 0]
    return float(-(py * np.log(py)).sum())


In [14]:
# Cell 5: Bigram MI against the within-review shuffle null

order = {}
for name, _, _ in GRANULARITIES:
    chains = loaded[name]["chains"]
    vocab  = sorted({e for c in chains for e in c})
    idx    = {e: i for i, e in enumerate(vocab)}
    ints   = [np.array([idx[e] for e in c], dtype=np.int64) for c in chains]

    obs_tbl = bigram_table(ints, len(vocab))
    obs_mi  = mutual_information(obs_tbl)
    h_y     = successor_entropy(obs_tbl)

    rng  = np.random.default_rng(RANDOM_STATE)
    null = np.empty(N_PERMUTATIONS)
    for i in range(N_PERMUTATIONS):
        null[i] = mutual_information(bigram_table([rng.permutation(c) for c in ints],
                                                  len(vocab)))

    excess = obs_mi - null.mean()
    frac   = excess / h_y
    z      = excess / null.std(ddof=1)
    p_perm = (1 + int((null >= obs_mi).sum())) / (N_PERMUTATIONS + 1)
    order[name] = {"mi_excess_frac_of_H": frac, "mi_z": z, "mi_p_perm": p_perm,
                   "n_types": len(vocab)}

    # z / p_perm say whether it is distinguishable from zero at all.
    print(f"{name:14s} MI {obs_mi:.5f}  null {null.mean():.5f}  excess {excess:+.5f} "
          f"= {frac:.4f} of H(Y)   z={z:6.1f}  p_perm={p_perm:.3f}")

clusters_30    MI 0.00658  null 0.00396  excess +0.00262 = 0.0010 of H(Y)   z=  14.7  p_perm=0.005
MR_4           MI 0.00009  null 0.00005  excess +0.00003 = 0.0000 of H(Y)   z=   1.3  p_perm=0.100
Agency_6       MI 0.00064  null 0.00016  excess +0.00048 = 0.0003 of H(Y)   z=  11.0  p_perm=0.005


## 3. Test 2: are events usually located at the same position?

In [15]:
# Cell 5: Mean relative position per event type

position = {}
for name, _, _ in GRANULARITIES:
    rows = []
    for c in loaded[name]["chains"]:
        L = len(c)
        if L < MIN_CHAIN_LEN:
            continue
        for i, e in enumerate(c):
            rows.append({"type": e, "rel_pos": i / (L - 1)})
    df = pd.DataFrame(rows)
    df["bin"] = np.minimum((df["rel_pos"] * N_POSITION_BINS).astype(int),
                           N_POSITION_BINS - 1)

    tab = pd.crosstab(df["type"], df["bin"])
    chi2, p_chi, _, _ = scipy.stats.chi2_contingency(tab)
    v = np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1)))

    means   = df.groupby("type")["rel_pos"].mean().sort_values()
    rng_pos = float(means.max() - means.min())
    position[name] = {"cramers_v": float(v), "chi2_p": float(p_chi),
                      "position_range": rng_pos, "means": means}

    effect = "small+" if v > CRAMERS_V_SMALL else "negligible"
    print(f"{name:14s} position range {means.min():.3f} to {means.max():.3f} "
          f"= {rng_pos:.3f}   Cramer's V {v:.4f} (Cohen small={CRAMERS_V_SMALL})  "
          f"chi2 p={p_chi:.3f}   [{effect}]")

clusters_30    position range 0.459 to 0.565 = 0.105   Cramer's V 0.0345 (Cohen small=0.1)  chi2 p=0.000   [negligible]
MR_4           position range 0.487 to 0.505 = 0.018   Cramer's V 0.0240 (Cohen small=0.1)  chi2 p=0.000   [negligible]
Agency_6       position range 0.490 to 0.514 = 0.024   Cramer's V 0.0272 (Cohen small=0.1)  chi2 p=0.000   [negligible]


## 4. Does the dependency survive distance?

In [16]:
# Cell 6: Does the dependency survive distance?
# skip-k pairs positions (i, i+k+1)
# The same within-review shuffle null applies at every k.

SKIP_GAPS = [0, 1, 2, 3, 5]

def gap_table(int_chains, n_types, k):
    tbl = np.zeros((n_types, n_types), dtype=np.int64)
    for c in int_chains:
        if len(c) < k + 2:
            continue
        np.add.at(tbl, (c[:-(k + 1)], c[k + 1:]), 1)
    return tbl


decay = {}
for name, _, _ in GRANULARITIES:
    chains = loaded[name]["chains"]
    vocab  = sorted({e for c in chains for e in c})
    idx    = {e: i for i, e in enumerate(vocab)}
    ints   = [np.array([idx[e] for e in c], dtype=np.int64) for c in chains]

    obs_mi, h_y = {}, {}
    for k in SKIP_GAPS:
        t = gap_table(ints, len(vocab), k)
        obs_mi[k] = mutual_information(t)
        h_y[k]    = successor_entropy(t)

    rng  = np.random.default_rng(RANDOM_STATE)
    null = {k: np.empty(N_PERMUTATIONS) for k in SKIP_GAPS}
    for i in range(N_PERMUTATIONS):
        shuffled = [rng.permutation(c) for c in ints]
        for k in SKIP_GAPS:
            null[k][i] = mutual_information(gap_table(shuffled, len(vocab), k))

    decay[name] = {k: (obs_mi[k] - null[k].mean()) / h_y[k] for k in SKIP_GAPS}

    cells = "  ".join(f"k={k}: {decay[name][k]:+.5f}" for k in SKIP_GAPS)
    print(f"{name:14} {cells}")

clusters_30    k=0: +0.00103  k=1: +0.00025  k=2: +0.00011  k=3: +0.00010  k=5: +0.00003
MR_4           k=0: +0.00003  k=1: +0.00008  k=2: -0.00002  k=3: +0.00007  k=5: -0.00003
Agency_6       k=0: +0.00032  k=1: +0.00018  k=2: -0.00001  k=3: +0.00005  k=5: -0.00001


## 5. Are the frequent transitions frequent for any reason but frequency?

Pointwise mutual information (Church & Hanks, 1990) for each transition:

    PMI = log2( observed / expected )

`expected` is the mean count over N_PERMUTATIONS within-review shuffles, same null as Test 1. 
PMI = 0  : occurs exactly as often as event frequencies predict (chance)
PMI > 0  : follows more often than chance      PMI < 0 : avoided
PMI is unstable for rare pairs, so the ranking below applies a minimum count.

In [17]:
# Cell 7: Are the frequent transitions frequent for any reason but frequency?

PRIMARY           = GRANULARITIES[0][0]   # fine alphabet; nested schemes are too coarse
MIN_COUNT_FOR_PMI = 200
N_TOP             = 10

chains = loaded[PRIMARY]["chains"]
vocab  = sorted({e for c in chains for e in c})
idx    = {e: i for i, e in enumerate(vocab)}
ints   = [np.array([idx[e] for e in c], dtype=np.int64) for c in chains]

observed = gap_table(ints, len(vocab), 0).astype(float)

rng = np.random.default_rng(RANDOM_STATE)
expected = np.zeros_like(observed)
for _ in range(N_PERMUTATIONS):
    expected += gap_table([rng.permutation(c) for c in ints], len(vocab), 0)
expected /= N_PERMUTATIONS

rows = []
for i in range(len(vocab)):
    for j in range(len(vocab)):
        if observed[i, j] > 0 and expected[i, j] > 0:
            rows.append({
                "transition": f"{vocab[i]} -> {vocab[j]}",
                "count":    int(observed[i, j]),
                "share":    observed[i, j] / observed.sum(),
                "expected": expected[i, j],
                "pmi":      float(np.log2(observed[i, j] / expected[i, j])),
            })
pmi_df = pd.DataFrame(rows)

print(f"{PRIMARY}: {int(observed.sum()):,} transitions, {len(pmi_df):,} distinct pairs")
print()
print(f"MOST FREQUENT transitions (top {N_TOP}):")
top = pmi_df.nlargest(N_TOP, "count")
print(top[["transition", "count", "share", "expected", "pmi"]]
      .to_string(index=False,
                 formatters={"share": "{:.2%}".format, "expected": "{:,.0f}".format,
                             "count": "{:,}".format, "pmi": "{:+.3f}".format}))

clusters_30: 112,876 transitions, 897 distinct pairs

MOST FREQUENT transitions (top 10):
                                  transition  count share expected    pmi
Being & Presentation -> Being & Presentation 10,283 9.11%   10,515 -0.032
      Being & Presentation -> Generic Action  5,421 4.80%    5,370 +0.014
      Generic Action -> Being & Presentation  5,223 4.63%    5,371 -0.040
           Cognition -> Being & Presentation  3,013 2.67%    2,850 +0.080
           Being & Presentation -> Cognition  2,941 2.61%    2,854 +0.044
            Generic Action -> Generic Action  2,748 2.43%    2,781 -0.017
          Being & Presentation -> Initiation  2,619 2.32%    2,647 -0.015
          Initiation -> Being & Presentation  2,577 2.28%    2,648 -0.039
          Being & Presentation -> Perception  2,412 2.14%    2,415 -0.002
          Perception -> Being & Presentation  2,401 2.13%    2,415 -0.008


## 6. Summary

In [18]:
# Cell 8: One row per granularity

rows = []
for name, _, _ in GRANULARITIES:
    o, p = order[name], position[name]
    row = {
        "granularity":         name,
        "n_types":             o["n_types"],
        # Order (MI): effect size + detectability, no floor.
        "mi_excess_frac_of_H": round(o["mi_excess_frac_of_H"], 5),
        "mi_z":                round(o["mi_z"], 1),
        "mi_p_perm":           round(o["mi_p_perm"], 4),
        # Position: Cramer's V against Cohen's (1988) small-effect benchmark (0.10).
        "position_range":      round(p["position_range"], 4),
        "position_cramers_v":  round(p["cramers_v"], 4),
        "position_chi2_p":     round(p["chi2_p"], 4),
        "position_effect":     "small+" if p["cramers_v"] > CRAMERS_V_SMALL else "negligible",
    }
    row.update({f"mi_excess_gap{k}": round(decay[name][k], 5) for k in SKIP_GAPS})
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(SUMMARY_PATH, index=False)